# 04 — Conversation History

This notebook covers the **history** item from the project roadmap (see README): extending notebook 03's single-shot `ask()` into a multi-turn conversation. Follow-up questions from the operator often depend on what was already asked/answered — this notebook demonstrates threading `chat_history` through both retrieval (via history-aware query reformulation) and generation.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import VECTOR_DIR, COLLECTION_NAME

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this notebook.')


## 1. Load the existing vector database

No PDF ingestion or embedding should happen here. That work was already completed in notebook 02 and cached in Chroma.

In [2]:
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import build_retriever

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
retriever = build_retriever(vectorstore, k=5)
print('Persistent retriever loaded.')


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Persistent retriever loaded.


## 2. Why follow-up questions need history-aware retrieval

A follow-up like *"what if that fault code has already cleared?"* means nothing to a vector search on its own — the embedding of "that fault code" carries no equipment-specific signal. `factory_floor.rag.contextualize_question` rewrites the follow-up into a standalone question (using the conversation so far) **before** retrieval runs, so the search actually has something specific to match against. This only costs an extra LLM call on turns after the first — the very first question in a conversation is never rewritten.

In [3]:
from factory_floor.rag import ask, build_chat_history, get_llm

llm = get_llm()
turns = []


## 3. Turn 1 — initial question

In [4]:
question_1 = 'A SINAMICS G120 drive is repeatedly tripping. What should a technician inspect before deciding on a cause?'

turns.append(ask(question_1, retriever, llm, chat_history=build_chat_history(turns)))

print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


For a SINAMICS G120 drive that is repeatedly tripping, a technician should inspect the following before deciding on a cause:

1. Check the line supply voltage at the input terminals and verify the line supply voltage setting (parameter p0210) to ensure it is within permissible limits [SOURCE 4].

2. Inspect the infeed and its environment, including line supply, filters, reactors, and fuses, as an infeed fault can cause trips [SOURCE 1].

3. Verify the DC link voltage conditions:
   - Check for overvoltage or undervoltage conditions in the DC link.
   - Review if the DC link voltage controller is activated and configured correctly (parameters p1240, p1280).
   - Consider increasing ramp-down time (p1121) and setting rounding times (p1130, p1136) to relieve DC link voltage controller stress [SOURCE 2, SOURCE 3].

4. Examine the braking module or braking controller for faults or overloads, including temperature issues [SOURCE 1].

5. Check the motor load and ambient temperature, and verif

## 4. Turn 2 — an elliptical follow-up

The next question only makes sense in light of turn 1 — it never repeats "SINAMICS G120" or "tripping". We print the rewritten `standalone_question` so the reformulation step is visible, not just trusted.

In [5]:
question_2 = 'What if that fault code has already cleared by the time I check?'

turns.append(ask(question_2, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
If the SINAMICS G120 drive fault code has already cleared by the time you check, what diagnostic steps or tools can a technician use to retrieve historical fault information or identify the cause of the previous trip?

ANSWER:
If the fault code has already cleared by the time you check, the documentation does not provide a direct procedure for this scenario. However, it is advisable to:

- Review the fault history or event log in the drive's diagnostic interface to identify the last fault code and its timestamp.
- Check the supply voltage and wiring conditions as transient faults like line faults or supply voltage faults may have caused the trip and then cleared [SOURCE 4].
- Inspect the infeed and braking module conditions, as intermittent faults in these areas can cause temporary trips [SOURCE 2].
- Verify the DRIVE-CLiQ wiring and EMC compliance to rule out communication faults that may clear after a reset [SOURCE 2].
- Perform a safety co

## 5. Turn 3 — a second follow-up

One more hop, to confirm the conversation keeps working beyond a single follow-up.

In [6]:
question_3 = 'And which of those checks needs the drive powered down first?'

turns.append(ask(question_3, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
Which of the recommended inspections for a SINAMICS G120 drive that is repeatedly tripping—but whose fault code has already cleared—require the drive to be powered down before performing the check?

ANSWER:
From the retrieved documentation, the checks that require the drive to be powered down first include:

- Disconnecting the infeed unit from the line supply to allow precharging resistors to cool down before further inspection or work on the infeed [SOURCE 4].

- Checking or replacing hardware components such as the Control Unit, Motor Module, Power Module, or Sensor Module, which triggers the need for a partial acceptance test and implies powering down the drive for safe replacement [SOURCE 5].

- Inspecting or correcting DRIVE-CLiQ wiring and EMC-compliant installation should be done with the drive powered down to avoid communication faults and ensure safety [SOURCE 1].

Other checks like verifying line supply voltage, parameter settings,

## Milestone checkpoint

The pipeline is now:

`follow-up question → history-aware reformulation → retrieval → contextualized answer`

This is exactly what the Streamlit app's follow-up box (rendered below each answer) exposes to the operator. Per the README roadmap, **history** is done — vision, agents, memory and safety validation remain.